# DeepFaceLive — Kaggle headless inference test

This notebook checks still-image face swapping on Kaggle **without opening the desktop app and without renting a paid cloud pod**.

## Before running
1. Open this notebook in Kaggle.
2. In the notebook's **Settings / Session options**, choose a **GPU** accelerator.
3. Make sure **Internet** is enabled.
4. Run the cells from top to bottom. The first download includes about 730 MB of model parts; allow it time to finish.

The first test uses the project's sample images of fictional demo people. It writes a face-crop result to `/kaggle/working/swapped_face.png`. The test does not yet measure live-video speed.

In [ ]:
# Step 1 — Check that Kaggle gave this notebook a GPU.
!nvidia-smi

import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Step 2 — Download only the source code and model files needed for this test.
# Model weights are large; sparse checkout skips unrelated video demos and extra models.
from pathlib import Path
import subprocess

REPO_DIR = Path('/kaggle/working/DeepFaceLive')
BRANCH_URL = 'https://github.com/johnlilda2-ui/DeepFaceLive.git'
BRANCH = 'experiment/cloud-headless-webrtc'

if not (REPO_DIR / '.git').exists():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--depth', '1', '--sparse',
        '--single-branch', '--branch', BRANCH, BRANCH_URL, str(REPO_DIR)
    ], check=True)

# Include Python code plus only the 4 model weights this test uses.
patterns = [
    '/scripts/',
    '/xlib/',
    '/modelhub/__init__.py',
    '/modelhub/onnx/__init__.py',
    '/modelhub/onnx/*/*.py',
    '/modelhub/onnx/InsightFaceSwap/inswapper_128.onnx.part*',
    '/modelhub/onnx/InsightFaceSwap/w600k_r50.onnx.part*',
    '/modelhub/onnx/InsightFace2d106/InsightFace2D106.onnx',
    '/modelhub/onnx/YoloV5Face/YoloV5Face.onnx',
    '/doc/celebs/Rob_Doe/Rob_Doe.png',
    '/doc/celebs/Millie_Park/Millie_Park.png',
]
subprocess.run(['git', '-C', str(REPO_DIR), 'sparse-checkout', 'set', '--no-cone', *patterns], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)

required = [
    'modelhub/onnx/InsightFaceSwap/inswapper_128.onnx.part0',
    'modelhub/onnx/InsightFaceSwap/w600k_r50.onnx.part0',
    'modelhub/onnx/InsightFace2d106/InsightFace2D106.onnx',
    'modelhub/onnx/YoloV5Face/YoloV5Face.onnx',
    'doc/celebs/Rob_Doe/Rob_Doe.png',
    'doc/celebs/Millie_Park/Millie_Park.png',
]
missing = [p for p in required if not (REPO_DIR / p).is_file()]
if missing:
    raise FileNotFoundError('Files were not checked out: ' + ', '.join(missing))
print('Repository and required model files are ready:', REPO_DIR)

In [ ]:
# CELL 3 — Install ONNX Runtime plus its required CUDA/cuDNN libraries.
# Important: CUDA showing in the provider list does NOT mean its system libraries are installed.
# The earlier cell skipped this install because it saw CUDA in the list, causing silent CPU fallback.
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'
], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'onnxruntime-gpu[cuda,cudnn]==1.31.0', 'onnx', 'numexpr'
], check=True)

# A fresh process checks installed runtime libraries and prints diagnostic information.
verify_code = r'''
import onnxruntime as ort
print('ONNX Runtime:', ort.__version__)
print('Available providers:', ort.get_available_providers())
if not hasattr(ort, 'preload_dlls'):
    raise RuntimeError('This ONNX Runtime build has no preload_dlls() helper.')
ort.preload_dlls(directory='')
if hasattr(ort, 'print_debug_info'):
    ort.print_debug_info()
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA provider is not listed'
'''
check = subprocess.run([sys.executable, '-c', verify_code], text=True)
if check.returncode != 0:
    raise RuntimeError('CUDA dependency setup failed. Send me this cell output.')
print('CUDA libraries installed; the following smoke test will verify the actual model sessions use CUDA.')

In [ ]:
# Step 4 — Run the smoke test using two supplied sample images.
# --require-cuda prevents a silent CPU fallback.
import subprocess, sys
from IPython.display import display, Image

source_image = REPO_DIR / 'doc/celebs/Rob_Doe/Rob_Doe.png'
target_image = REPO_DIR / 'doc/celebs/Millie_Park/Millie_Park.png'
output_image = Path('/kaggle/working/swapped_face.png')
command = [
    sys.executable, str(REPO_DIR / 'scripts/headless_inference_smoke_test.py'),
    '--source-image', str(source_image),
    '--target-image', str(target_image),
    '--output', str(output_image),
    '--require-cuda',
]
print('Running:', ' '.join(command))
result = subprocess.run(command, cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError('The headless smoke test failed. Copy the [FAIL] line and traceback above and send them to ChatGPT.')
if not output_image.is_file():
    raise FileNotFoundError('Test reported success but output image was not found: ' + str(output_image))
print('PASS — output image created:', output_image, 'bytes:', output_image.stat().st_size)
display(Image(filename=str(output_image)))

## What to send back
If the notebook fails, send the **first red error / traceback** from the cell that stopped. Do not rent a GPU anywhere else and do not change the code yet. If the image appears and says `SMOKE TEST PASSED`, this confirms only still-image inference; real-time WebRTC streaming is the next separate step.